### Configurations

In [1]:
SEED = 0

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import random

# Network Settings
XM, YM = 100, 100
NUM_NODES = 100
ROUNDS = 60000

# Mobile Sink Trajectory
CENTER_X, CENTER_Y = 50, 50
TRAJ_RADIUS = 25.0
MS_SPEED = 1.0  # Degrees/round

# Energy Model (Standard WSN / LEACH-RN Paper)
INITIAL_ENERGY = 0.5
MIN_ENERGY = 0.05
E_ELEC = 50e-9       # 50 nJ/bit (Circuitry)
E_FS = 10e-12        # 10 pJ/bit/m^2 (Free Space)
E_MP = 0.0013e-12    # 0.0013 pJ/bit/m^4 (Multipath)
E_DA = 5e-9          # 5 nJ/bit/signal (Aggregation)
D0 = (E_FS / E_MP)**0.5  # ~87m

# Traffic & Realism
PACKET_SIZE_DATA = 4000
PACKET_SIZE_CTRL = 200
DATA_PROB = 0.11     # ~16,000 packets total
E_IDLE = 50e-6       # 50 µJ/round (Idle/Listening Cost)

# Fuzzy Constants
DESIRED_CH_PCT = 0.1

In [ ]:
def get_dist_matrix(p1, p2):
    """Euclidean distance between two sets of points (N,2) and (M,2)."""
    return np.sqrt(np.sum((p1[:, None, :] - p2[None, :, :]) ** 2, axis=-1))


def calc_tx_energy(dists, bits):
    """Vectorized Tx Energy: E_elec + Amp * dist^alpha"""
    energy = np.zeros_like(dists)
    mask_fs = dists < D0
    mask_mp = ~mask_fs

    # Free Space (d^2)
    energy[mask_fs] = bits * E_ELEC + bits * E_FS * (dists[mask_fs]**2)
    # Multipath (d^4)
    energy[mask_mp] = bits * E_ELEC + bits * E_MP * (dists[mask_mp]**4)
    return energy


def calc_rx_energy(bits):
    """Rx Energy: E_elec * bits"""
    return bits * E_ELEC


def check_packet_loss(dists):
    """
    Realistic Packet Loss Model
    Prob = min(0.4, 0.1 + 0.3 * (dist / 87))
    """
    probs = np.minimum(0.1, 0.1 + 0.3 * (dists / 87.0))
    return np.random.rand(len(dists)) < probs

### Fuzzy

In [ ]:
class VectorizedFuzzyLogic:
    def __init__(self):
        # Membership Functions (Normalized inputs [0, 1])
        # Based on Article Figures 7 & 8
        self.low = np.array([0.0, 0.0, 0.4])
        self.med = np.array([0.2, 0.5, 0.8])
        self.high = np.array([0.6, 1.0, 1.0])

    def trimf(self, x, abc):
        """Vectorized Triangular Membership Function"""
        a, b, c = abc
        term1 = (x - a) / (b - a + 1e-9)
        term2 = (c - x) / (c - b + 1e-9)
        return np.maximum(0, np.minimum(term1, term2))

    def compute_ch_prob(self, E, Den, Dist):
        """
        Fuzzy CH Selection (Section 3.4)
        Inputs: Residual Energy (High), Node Density (High), Dist to MS (Low)
        """
        # Fuzzify
        e_H = self.trimf(E, self.high)
        e_M = self.trimf(E, self.med)
        e_L = self.trimf(E, self.low)

        den_H = self.trimf(Den, self.high)
        den_M = self.trimf(Den, self.med)

        d_L = self.trimf(Dist, self.low)
        d_M = self.trimf(Dist, self.med)
        d_H = self.trimf(Dist, self.high)

        # Rules (Inferred from Logic: Better stats -> Higher Prob)
        # 1. Very High: High Energy + High Density + Near Sink
        r_vhigh = np.minimum(np.minimum(e_H, den_H), d_L)

        # 2. High: High Energy + Med Density + Med Sink
        r_high = np.minimum(np.minimum(e_H, den_M), d_M)

        # 3. Medium: Med Energy + Med Density
        r_med = np.minimum(e_M, den_M)

        # 4. Low: Low Energy OR Far Sink
        r_low = np.maximum(e_L, d_H)

        # Defuzzification (Weighted Average)
        num = r_vhigh * 0.9 + r_high * 0.75 + r_med * 0.5 + r_low * 0.1
        den = r_vhigh + r_high + r_med + r_low + 1e-9
        return num / den

    def compute_rn_suitability(self, E, DistTraj, Centrality):
        """
        Fuzzy RN Selection (Section 3.2 - 3.3)
        Inputs: Residual Energy (High), Dist to Trajectory (Low), Centrality (Low=Central)
        """
        # Fuzzify
        e_H = self.trimf(E, self.high)
        e_M = self.trimf(E, self.med)
        e_L = self.trimf(E, self.low)

        dt_L = self.trimf(DistTraj, self.low)   # Near Trajectory
        dt_M = self.trimf(DistTraj, self.med)
        dt_H = self.trimf(DistTraj, self.high)  # Far

        c_L = self.trimf(Centrality, self.low)  # High Centrality (Low Dist)
        c_M = self.trimf(Centrality, self.med)
        c_H = self.trimf(Centrality, self.high)  # Low Centrality

        # Rules (Based on Table 2)
        # 1. High Suitability
        r_high = np.minimum(np.minimum(e_H, dt_L), c_L)

        # 2. Medium Suitability
        r_med = np.minimum(e_M, dt_M)

        # 3. Low Suitability
        r_low = np.maximum(e_L, dt_H)

        # Defuzzification
        num = r_high * 0.9 + r_med * 0.5 + r_low * 0.1
        den = r_high + r_med + r_low + 1e-9
        return num / den

### LEACH

In [5]:
class FuzzyLEACH_RN_Sim:
    def __init__(self):
        # Init Nodes
        np.random.seed(SEED)
        self.coords = np.random.rand(NUM_NODES, 2) * [XM, YM]
        self.energies = np.full(NUM_NODES, INITIAL_ENERGY)
        self.alive = np.ones(NUM_NODES, dtype=bool)

        # Mobile Sink
        self.ms_angle = 0.0
        self.ms_pos = np.array([CENTER_X + TRAJ_RADIUS, CENTER_Y])

        # Logic
        self.flc = VectorizedFuzzyLogic()
        self.ch_ids = []
        self.rn_ids = []  # List of RNs
        self.cluster_labels = np.full(NUM_NODES, -1)

        # Statistics
        self.stats = {
            'gen': 0,
            'del': 0,
            'alive': [],
            'energy': [],
            'fnd': None, 'hnd': None, 'lnd': None
        }

    def update_ms_position(self):
        """Move Mobile Sink along circular trajectory"""
        rad = np.radians(self.ms_angle)
        self.ms_pos[0] = CENTER_X + TRAJ_RADIUS * np.cos(rad)
        self.ms_pos[1] = CENTER_Y + TRAJ_RADIUS * np.sin(rad)
        self.ms_angle = (self.ms_angle + MS_SPEED) % 360

    def clustering_phase(self):
        """Fuzzy Cluster Head Selection"""
        alive_idx = np.where(self.alive)[0]
        if len(alive_idx) == 0:
            return

        #  1. Control Overhead: Neighbor Discovery
        # All nodes broadcast hello (Range ~30m)
        # Tx Cost
        tx_cost = calc_tx_energy(
            np.full(len(alive_idx), 30.0), PACKET_SIZE_CTRL)
        # Rx Cost (Avg 5 neighbors)
        rx_cost = calc_rx_energy(PACKET_SIZE_CTRL * 5)
        self.energies[alive_idx] -= (tx_cost + rx_cost)

        #  2. Calculate Fuzzy Inputs
        # Energy (Normalized)
        e_norm = self.energies[alive_idx] / INITIAL_ENERGY

        # Dist to MS (Normalized)
        d_ms = np.linalg.norm(self.coords[alive_idx] - self.ms_pos, axis=1)
        d_norm = np.clip(d_ms / 141.0, 0, 1)

        # Density (Neighbors / 20)
        dists_mat = get_dist_matrix(
            self.coords[alive_idx], self.coords[alive_idx])
        neighbors = np.sum(dists_mat < 30.0, axis=1)
        den_norm = np.clip(neighbors / 20.0, 0, 1)

        #  3. Compute Probabilities
        probs = self.flc.compute_ch_prob(e_norm, den_norm, d_norm)

        #  4. Elect CHs
        # Select top P% nodes
        k = max(1, int(NUM_NODES * DESIRED_CH_PCT))
        sorted_idx = np.argsort(-probs)
        self.ch_ids = alive_idx[sorted_idx[:k]].tolist()

        #  5. Cluster Formation (Overhead)
        if self.ch_ids:
            # CHs Broadcast Advertisement
            # Range ~87m to cover area
            adv_tx = calc_tx_energy(
                np.full(len(self.ch_ids), 87.0), PACKET_SIZE_CTRL)
            self.energies[self.ch_ids] -= adv_tx

            # Non-CHs join nearest CH
            non_chs = np.setdiff1d(alive_idx, self.ch_ids)
            if len(non_chs) > 0:
                # Rx Advertisement
                self.energies[non_chs] -= calc_rx_energy(
                    PACKET_SIZE_CTRL * len(self.ch_ids))

                # Find nearest CH
                ch_coords = self.coords[self.ch_ids]
                dists_to_ch = get_dist_matrix(self.coords[non_chs], ch_coords)
                nearest_idx = np.argmin(dists_to_ch, axis=1)

                # Tx Join Request
                join_dists = dists_to_ch[np.arange(len(non_chs)), nearest_idx]
                self.energies[non_chs] -= calc_tx_energy(
                    join_dists, PACKET_SIZE_CTRL)

                # CH Rx Join (Approx)
                # We skip charging CH Rx for every single join for speed/simplicity,
                # or add avg cost. Let's add avg cost.
                avg_rx_join = calc_rx_energy(
                    PACKET_SIZE_CTRL * (len(non_chs)/len(self.ch_ids)))
                self.energies[self.ch_ids] -= avg_rx_join

                # Assign Clusters
                self.cluster_labels[non_chs] = np.array(self.ch_ids)[
                    nearest_idx]
                self.cluster_labels[self.ch_ids] = self.ch_ids  # Self

    def rn_selection_phase(self):
        """Fuzzy Rendezvous Node Selection (Algorithm 1)"""
        if not self.ch_ids:
            return
        ch_idx = np.array(self.ch_ids)

        #  1. Overhead: CHs Exchange Info
        # Tx/Rx among CHs to calculate centrality
        if len(ch_idx) > 1:
            # Tx Status (Range ~100m)
            self.energies[ch_idx] -= calc_tx_energy(
                np.full(len(ch_idx), 100.0), PACKET_SIZE_CTRL)
            # Rx Status
            self.energies[ch_idx] -= calc_rx_energy(
                PACKET_SIZE_CTRL * (len(ch_idx)-1))

        #  2. Fuzzy Inputs
        e_norm = self.energies[ch_idx] / INITIAL_ENERGY

        # Dist to Trajectory (R=30, Center=50,50)
        d_center = np.linalg.norm(
            self.coords[ch_idx] - [CENTER_X, CENTER_Y], axis=1)
        d_traj = np.abs(d_center - TRAJ_RADIUS)
        dt_norm = np.clip(d_traj / 50.0, 0, 1)

        # Centrality (Avg Dist to other CHs)
        if len(ch_idx) > 1:
            ch_dists = get_dist_matrix(
                self.coords[ch_idx], self.coords[ch_idx])
            c_norm = np.clip(np.mean(ch_dists, axis=1) / 100.0, 0, 1)
        else:
            c_norm = np.zeros(len(ch_idx))

        #  3. Compute Suitability
        suitability = self.flc.compute_rn_suitability(e_norm, dt_norm, c_norm)

        #  4. Select RNs
        # Algorithm: Each CH selects an RN.
        # We designate the Top 30% of CHs as "Active RNs"
        n_rn = max(1, int(len(ch_idx) * 0.3))
        sorted_rn_idx = np.argsort(-suitability)
        self.rn_ids = ch_idx[sorted_rn_idx[:n_rn]]

    def run(self):

        for r in range(1, ROUNDS + 1):
            n_alive = np.sum(self.alive)
            if n_alive == 0:
                self.stats['lnd'] = r
                break

            # Metrics
            if self.stats['fnd'] is None and n_alive < NUM_NODES:
                self.stats['fnd'] = r
            if self.stats['hnd'] is None and n_alive <= NUM_NODES/2:
                self.stats['hnd'] = r
            if self.stats['lnd'] is None and n_alive == 0:
                self.stats['lnd'] = r

            self.stats['alive'].append(n_alive)
            self.stats['energy'].append(np.mean(self.energies))

            # 1. Move Sink
            self.update_ms_position()

            # 2. Control Phase (Clustering + RN)
            self.clustering_phase()
            self.rn_selection_phase()

            # 3. Data Transmission Phase

            # A. SN -> CH (Single Hop)
            gen_mask = (np.random.rand(NUM_NODES) < DATA_PROB) & self.alive & ~np.isin(
                np.arange(NUM_NODES), self.ch_ids)
            senders = np.where(gen_mask)[0]

            # Buffer for packets at CHs
            ch_buffer = {ch: 0 for ch in self.ch_ids}

            if len(senders) > 0:
                self.stats['gen'] += len(senders)

                targets = self.cluster_labels[senders]
                # Filter valid targets
                valid = targets != -1
                senders = senders[valid]
                targets = targets[valid]

                if len(senders) > 0:
                    s_pos = self.coords[senders]
                    t_pos = self.coords[targets]
                    dists = np.sqrt(np.sum((s_pos - t_pos)**2, axis=1))

                    # Tx Energy (SN)
                    self.energies[senders] -= calc_tx_energy(
                        dists, PACKET_SIZE_DATA)

                    # Packet Loss Check
                    lost = check_packet_loss(dists)
                    rec_chs = targets[~lost]

                    # Rx Energy (CH)
                    uni_chs, counts = np.unique(rec_chs, return_counts=True)
                    for ch, cnt in zip(uni_chs, counts):
                        if self.alive[ch]:
                            self.energies[ch] -= calc_rx_energy(
                                PACKET_SIZE_DATA * cnt)
                            ch_buffer[ch] += cnt

            # B. CH -> RN -> MS (Multi-Hop)
            if len(self.rn_ids) > 0:
                rn_coords = self.coords[self.rn_ids]

                for ch in self.ch_ids:
                    if not self.alive[ch]:
                        continue

                    # Own Traffic
                    if np.random.rand() < DATA_PROB:
                        self.stats['gen'] += 1
                        ch_buffer[ch] += 1

                    pkts = ch_buffer[ch]
                    if pkts == 0:
                        continue

                    # Aggregation Energy
                    self.energies[ch] -= (E_DA * PACKET_SIZE_DATA * pkts)

                    # Select Best RN (Nearest)
                    dists_rn = np.linalg.norm(
                        self.coords[ch] - rn_coords, axis=1)
                    best_rn_idx = np.argmin(dists_rn)
                    target_rn = self.rn_ids[best_rn_idx]
                    dist_to_rn = dists_rn[best_rn_idx]

                    # CH -> RN Tx
                    # (If CH is RN, distance is 0, cost is 0)
                    if ch != target_rn:
                        self.energies[ch] -= calc_tx_energy(
                            np.array([dist_to_rn]), PACKET_SIZE_DATA)[0]

                        # Loss Check
                        if check_packet_loss(np.array([dist_to_rn]))[0]:
                            continue

                        # RN Rx
                        if self.alive[target_rn]:
                            self.energies[target_rn] -= calc_rx_energy(
                                PACKET_SIZE_DATA)
                        else:
                            continue  # RN Dead

                    # RN -> MS Tx
                    d_ms = np.linalg.norm(self.coords[target_rn] - self.ms_pos)
                    e_tx_ms = calc_tx_energy(
                        np.array([d_ms]), PACKET_SIZE_DATA)[0]
                    self.energies[target_rn] -= e_tx_ms

                    if not check_packet_loss(np.array([d_ms]))[0]:
                        self.stats['del'] += pkts

            # 4. Idle Energy
            self.energies[self.alive] -= E_IDLE

            # 5. Update Dead
            self.alive = self.energies > MIN_ENERGY
            self.energies[~self.alive] = 0.0

            # if r % 200 == 0:
            #     print(f"Round {r}: Alive={n_alive} | Gen={self.stats['gen']} | Del={self.stats['del']}")

        # Final Results
        pdr = (self.stats['del'] / max(1, self.stats['gen'])) * 100
        print("FINAL RESULTS (Fuzzy LEACH-RN-MS)")
        print(f"Total Packets Generated : {self.stats['gen']}")
        print(f"Total Packets Delivered : {self.stats['del']}")
        print(f"PDR                     : {pdr:.2f}%")
        print(f"FND (First Node Dead)   : {self.stats['fnd']}")
        print(f"HND (Half Node Dead)    : {self.stats['hnd']}")
        print(f"LND (Last Node Dead)    : {self.stats['lnd']}")

        # plt.figure(figsize=(10, 4))
        # plt.subplot(1, 2, 1)
        # plt.plot(self.stats['alive'])
        # plt.title("Alive Nodes")
        # plt.subplot(1, 2, 2)
        # plt.plot(self.stats['energy'], color='green')
        # plt.title("Avg Energy")
        # plt.tight_layout()
        # plt.show()

### Run

In [6]:
for i in range(31):
    SEED = i
    print(f"\nSEED: {SEED}")
    sim = FuzzyLEACH_RN_Sim()
    sim.run()


SEED: 0
FINAL RESULTS (Fuzzy LEACH-RN-MS)
Total Packets Generated : 15902
Total Packets Delivered : 12061
PDR                     : 75.85%
FND (First Node Dead)   : 964
HND (Half Node Dead)    : 1447
LND (Last Node Dead)    : 1827

SEED: 1
FINAL RESULTS (Fuzzy LEACH-RN-MS)
Total Packets Generated : 15857
Total Packets Delivered : 12066
PDR                     : 76.09%
FND (First Node Dead)   : 857
HND (Half Node Dead)    : 1514
LND (Last Node Dead)    : 1827

SEED: 2
FINAL RESULTS (Fuzzy LEACH-RN-MS)
Total Packets Generated : 16013
Total Packets Delivered : 12252
PDR                     : 76.51%
FND (First Node Dead)   : 875
HND (Half Node Dead)    : 1508
LND (Last Node Dead)    : 1830

SEED: 3
FINAL RESULTS (Fuzzy LEACH-RN-MS)
Total Packets Generated : 15737
Total Packets Delivered : 12028
PDR                     : 76.43%
FND (First Node Dead)   : 800
HND (Half Node Dead)    : 1559
LND (Last Node Dead)    : 1846

SEED: 4
FINAL RESULTS (Fuzzy LEACH-RN-MS)
Total Packets Generated : 160